# 02 · Entrenamiento, calibración y exportación

Entrena la red, la calibra con la partición de validación y exporta en `models/` el modelo, el preprocesador, el calibrador, las variables y la Cartera puntuada. Toda la lógica vive en `src/` (ADR-0001).

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Rama que se entrena en Colab. Mientras el slice no esté en main, usa su rama
# (tiene que estar publicada en GitHub con git push).
BRANCH = "vs-002-entrenar-calibrar-exportar"
REPO_URL = "https://github.com/Emmanuel2503/deep-retention.git"


def git(*args):
    subprocess.run(["git", *args], check=True)


if "google.colab" in sys.modules:
    repo = Path("/content/deep-retention")
    try:
        if not repo.exists():
            git("clone", "--branch", BRANCH, REPO_URL, str(repo))
        else:
            # Una sesión anterior dejó el clon: se actualiza a la última versión de BRANCH.
            git("-C", str(repo), "fetch", "origin", BRANCH)
            git("-C", str(repo), "checkout", BRANCH)
            git("-C", str(repo), "pull", "--ff-only", "origin", BRANCH)
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            f"No se pudo obtener la rama {BRANCH!r} de {REPO_URL}. "
            "Comprueba que la rama esté publicada en GitHub (git push -u origin <rama>)."
        ) from error
    %cd {repo}
    !git log -1 --oneline
    %pip install -q -r requirements.txt
elif Path.cwd().name == "notebooks":
    %cd ..

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

## Versiones del entorno

Se fijan en `requirements.txt` con `==` (SRS/TR-02). Colab debe usar Python 3.12.

In [ ]:
import platform
from importlib.metadata import version

print("python", platform.python_version())
for lib in ["tensorflow", "scikit-learn", "joblib", "numpy", "pandas"]:
    print(f"{lib}=={version(lib)}")

## Particiones reproducibles (AC-01)

In [ ]:
from src.preprocessing import TARGET, load_dataset, split_data

df = load_dataset()
run_a = split_data(df)
run_b = split_data(df)
for name, a, b in zip(["train", "val", "test"], run_a, run_b):
    same = set(a["CustomerId"]) == set(b["CustomerId"])
    print(name, len(a), "abandono", round(a[TARGET].mean(), 4), "mismos clientes:", same)

## Entrenamiento

In [ ]:
from src.train import run_training

result = run_training()
print("mejor época:", result.best_epoch, "· detenido en:", result.stopped_epoch)

## Comprobación del early stopping (AC-02)

In [ ]:
from src.config import MODELS_DIR
from src.inference import ChurnPredictor
from src.model import PATIENCE
from src.preprocessing import TARGET

predictor = ChurnPredictor.load()
val = df.loc[df["CustomerId"].isin(result.split_ids["val"])]
val_loss_saved = float(
    predictor.model.evaluate(predictor.transform(val), val[TARGET], verbose=0)
)
val_loss_best = result.val_loss[result.best_epoch]
print("PATIENCE:", PATIENCE)
print("stopped_epoch - best_epoch:", result.stopped_epoch - result.best_epoch)
print(f"val_loss del historial en best_epoch: {val_loss_best:.6f}")
print(f"val_loss del modelo guardado:          {val_loss_saved:.6f}")
print("diferencia:", abs(val_loss_best - val_loss_saved))

## Predicción sin reentrenar (AC-03)

In [ ]:
import pandas as pd

p = predictor.predict(df)
print(len(p), "clientes · min", p.min(), "· max", p.max())
pd.read_csv(MODELS_DIR / "portfolio_scored.csv").head()